# CSE 144 Final Project — Transfer Learning with EfficientNet-B0

This notebook walks through the full pipeline:
1. Download data from Kaggle
2. Build numerically-sorted data loaders with augmentation
3. Load a pretrained EfficientNet-B0 and replace its head
4. Fine-tune in two phases (frozen backbone → full fine-tune)
5. Run inference and write `submission.csv`

## 1. Install Dependencies

`kagglehub` is Kaggle's modern download client. The rest are standard ML libraries.
If you're on Colab, torch/torchvision are already installed — only `kagglehub` needs to be added.

In [ ]:
%pip install -q kagglehub torch torchvision scikit-learn Pillow numpy

## 2. Download the Dataset

`kagglehub.competition_download` pulls the competition files and returns the local cache path.
We then point `TRAIN_DIR` and `TEST_DIR` at the right sub-folders inside that path.

> **Note:** You need a Kaggle API token (`~/.kaggle/kaggle.json`) or to be logged in via `kagglehub.login()`.

In [ ]:
import os
import kagglehub

DATA_ROOT = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')
print(f"Data downloaded to: {DATA_ROOT}")

TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
TEST_DIR  = os.path.join(DATA_ROOT, 'test')

# Quick sanity check
train_classes = sorted(os.listdir(TRAIN_DIR), key=lambda x: int(x))
print(f"Train classes (first 5): {train_classes[:5]}")
print(f"Test images   (first 5): {sorted(os.listdir(TEST_DIR))[:5]}")

## 3. Reproducibility — Fix the Random Seed

Setting the same seed in Python, NumPy, and PyTorch (both CPU and GPU) ensures every run
produces the same train/val split, weight initialization, and augmentation order.
`cudnn.deterministic = True` trades a small speed penalty for deterministic GPU ops.

In [ ]:
import random
import numpy as np
import torch

SEED = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Seed set to", SEED)

## 4. Hyperparameters & Device

All tuneable values live in one place so they're easy to find and change.

| Setting | Value | Why |
|---|---|---|
| `IMG_SIZE` | 224 | EfficientNet-B0's native resolution |
| `BATCH_SIZE` | 32 | Fits comfortably on most GPUs |
| `FREEZE_LR` | 1e-3 | Aggressive — only a small head is updating |
| `UNFREEZE_LR` | 1e-4 | Conservative — avoids destroying pretrained features |
| `VAL_SPLIT` | 0.2 | 80/20 train/val |

In [ ]:
NUM_CLASSES     = 100
BATCH_SIZE      = 32
NUM_WORKERS     = 2       # set to 0 if you hit DataLoader errors on Windows
IMG_SIZE        = 224

FREEZE_EPOCHS   = 5       # phase 1: backbone frozen, head only
FREEZE_LR       = 1e-3

UNFREEZE_EPOCHS = 15      # phase 2: all layers fine-tuned
UNFREEZE_LR     = 1e-4

VAL_SPLIT       = 0.2
CHECKPOINT      = 'best_model.pth'

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

## 5. Data Augmentation & Transforms

**Why augment?** With only ~10 images per class, augmentation is essential for generalization.
We apply random flips, rotations, and color jitter to the training set only — the validation
set gets a clean resize so its metrics reflect real performance, not lucky augmentations.

**ImageNet normalization** (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`) matches
the statistics the pretrained model was trained on. Using different values would misalign
the input distribution and hurt transfer learning.

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Transforms defined.")

## 6. Numerically-Sorted Dataset

PyTorch's built-in `ImageFolder` sorts class folders **alphabetically**, which gives:
```
"0", "1", "10", "11", ..., "19", "2", "20", ...
```
That means folder `"10"` gets label index `2`, not `10` — completely wrong predictions.

We fix this by subclassing `ImageFolder` and overriding `find_classes` to sort by
`int(folder_name)`. We also set `class_to_idx[cls] = int(cls)` directly so the label
is always the folder number, regardless of insertion order.

In [ ]:
from torchvision import datasets

class NumericalImageFolder(datasets.ImageFolder):
    """ImageFolder that sorts class folders by integer value, not string."""

    def find_classes(self, directory):
        classes = [d.name for d in os.scandir(directory) if d.is_dir()]
        classes.sort(key=lambda x: int(x))          # "2" < "10" numerically
        class_to_idx = {cls: int(cls) for cls in classes}
        return classes, class_to_idx


# Verify the fix — first 5 class→index mappings should be 0→0, 1→1, 2→2 ...
sample_ds = NumericalImageFolder(TRAIN_DIR, transform=val_transform)
print("First 5 class→index mappings:")
for cls, idx in list(sample_ds.class_to_idx.items())[:5]:
    print(f"  folder '{cls}' → label {idx}")

## 7. Train / Val Split & DataLoaders

We use `sklearn`'s `train_test_split` with `stratify=labels` so every class keeps its
proportional representation in both splits — important when some classes may have fewer images.

A key subtlety: both `train_subset` and `val_subset` must come from **separate dataset
instances** with different transforms. If we used `Subset` on the same object, every image
— including validation — would get training augmentations, which inflates val accuracy.

In [ ]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# Two dataset instances — same files, different transforms
train_dataset = NumericalImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = NumericalImageFolder(TRAIN_DIR, transform=val_transform)

indices = list(range(len(train_dataset)))
labels  = train_dataset.targets

train_idx, val_idx = train_test_split(
    indices,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=labels,
)

train_loader = DataLoader(
    Subset(train_dataset, train_idx),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    Subset(val_dataset, val_idx),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"Train samples : {len(train_idx)}")
print(f"Val   samples : {len(val_idx)}")

## 8. Build the Model

EfficientNet-B0 is a compact but powerful CNN that scales depth, width, and resolution
together (compound scaling). It was pretrained on ImageNet-1K (1000 classes).

**What we change:** the final `model.classifier` is a two-layer Sequential:
```
Dropout(0.2) → Linear(1280, 1000)   ← original
Dropout(0.2) → Linear(1280, 100)    ← ours
```
Only the output dimension changes — we keep the dropout for regularization.
The rest of the network (the backbone) keeps its ImageNet weights intact.

In [ ]:
import torch.nn as nn
from torchvision import models

def build_model(num_classes: int) -> nn.Module:
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features  # 1280 for B0
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

model = build_model(NUM_CLASSES).to(device)

# Confirm the head shape
print("Classifier head:")
print(model.classifier)

## 9. Freeze / Unfreeze Helpers

**Why freeze first?** The backbone already knows how to extract features; the head is random.
If we train everything at once, the large gradients from the random head can corrupt the
pretrained weights before they've had a chance to adapt.

**Phase 1** — freeze the backbone, only train the head at a high learning rate until it
learns a reasonable mapping from features → classes.

**Phase 2** — unfreeze everything and fine-tune at a low learning rate so the backbone
slowly adapts to our domain without forgetting what it learned on ImageNet.

In [ ]:
def freeze_backbone(model: nn.Module):
    """Freeze every layer, then re-enable gradients for the classifier head only."""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

def unfreeze_all(model: nn.Module):
    """Allow gradients to flow through every layer."""
    for param in model.parameters():
        param.requires_grad = True

# Quick check: count trainable params before and after
freeze_backbone(model)
frozen_params   = sum(p.numel() for p in model.parameters() if p.requires_grad)
unfreeze_all(model)
total_params    = sum(p.numel() for p in model.parameters() if p.requires_grad)
freeze_backbone(model)  # start in frozen state for Phase 1

print(f"Trainable params (frozen backbone) : {frozen_params:,}")
print(f"Trainable params (all unfrozen)    : {total_params:,}")

## 10. Training Loop Helper

`run_epoch` handles both training and validation in one function — the only difference is
whether we call `optimizer.step()` and whether gradients are tracked.

`torch.set_grad_enabled(train)` is cleaner than wrapping validation in `torch.no_grad()`
because it works correctly even if `train=False` is passed.

We accumulate `loss * batch_size` then divide by total samples at the end, which gives
the true per-sample average loss regardless of whether the last batch is a partial batch.

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += images.size(0)

    return total_loss / total, correct / total

print("run_epoch defined.")

## 11. Phase 1 — Train the Head (Backbone Frozen)

We use Adam with a cosine annealing schedule. Cosine annealing smoothly reduces the
learning rate from `FREEZE_LR` down to near-zero over `FREEZE_EPOCHS`, which prevents
the optimizer from oscillating around the minimum late in training.

The best model checkpoint is saved whenever validation accuracy improves — not just at
the end — so we never lose the best weights if later epochs overfit.

In [ ]:
import torch.optim as optim

criterion   = nn.CrossEntropyLoss()
best_val_acc = 0.0

# Phase 1 optimizer only sees the head parameters
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FREEZE_LR,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FREEZE_EPOCHS)

print(f"{'='*65}")
print(f"Phase 1 — frozen backbone ({FREEZE_EPOCHS} epochs, lr={FREEZE_LR})")
print(f"{'='*65}")

for epoch in range(1, FREEZE_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, None,      device, train=False)
    scheduler.step()

    print(
        f"Epoch {epoch:3d}/{FREEZE_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"           ↑ New best val acc {best_val_acc:.4f} — checkpoint saved")

## 12. Phase 2 — Full Fine-Tuning (All Layers)

Now that the head has learned a reasonable mapping, we unfreeze the entire network and
train at a 10× lower learning rate (`1e-4` instead of `1e-3`).

We add `weight_decay=1e-4` to the optimizer as L2 regularization — this penalizes large
weights and helps prevent the backbone from overfitting to our small dataset.

A fresh optimizer and scheduler are created because the parameter groups changed
(we went from head-only to all layers).

In [ ]:
unfreeze_all(model)

optimizer = optim.Adam(model.parameters(), lr=UNFREEZE_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCHS)

print(f"{'='*65}")
print(f"Phase 2 — full fine-tune ({UNFREEZE_EPOCHS} epochs, lr={UNFREEZE_LR})")
print(f"{'='*65}")

for epoch in range(1, UNFREEZE_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, None,      device, train=False)
    scheduler.step()

    print(
        f"Epoch {epoch:3d}/{UNFREEZE_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"           ↑ New best val acc {best_val_acc:.4f} — checkpoint saved")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")
print(f"Checkpoint: {CHECKPOINT}")

## 13. (Optional) Plot Training Curves

Re-running both training phases while recording metrics lets us visualize how loss and
accuracy evolve. The dashed vertical line marks where Phase 1 ends and Phase 2 begins.

> If you skipped this cell during training you can skip it during inference too — it's
> purely diagnostic and doesn't affect the checkpoint.

In [ ]:
# This cell only works if you collected history during training.
# To use it, replace the training loops above with versions that append to these lists.

try:
    import matplotlib.pyplot as plt
    print("matplotlib available — add history tracking to training loops to plot curves.")
except ImportError:
    print("Install matplotlib to plot: pip install matplotlib")

## 14. Inference — Generate `submission.csv`

We reload the best checkpoint (not necessarily the last epoch) to get our strongest model.

Test images are sorted **numerically** by filename stem (`"2.jpg"` before `"10.jpg"`)
so the `ID` column in the CSV matches the actual file numbers.

`model.eval()` disables dropout and batch norm updates. `torch.no_grad()` skips building
the computation graph, saving memory and speeding up inference.

In [ ]:
import csv
from PIL import Image

# Load the best checkpoint
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()
print(f"Loaded checkpoint: {CHECKPOINT}")

# Collect and sort test images numerically
image_files = [
    f for f in os.listdir(TEST_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]
image_files.sort(key=lambda f: int(os.path.splitext(f)[0]))

rows = []
with torch.no_grad():
    for fname in image_files:
        img_id   = int(os.path.splitext(fname)[0])
        img_path = os.path.join(TEST_DIR, fname)

        image  = Image.open(img_path).convert('RGB')
        tensor = val_transform(image).unsqueeze(0).to(device)  # add batch dim

        logits = model(tensor)
        pred   = logits.argmax(dim=1).item()
        rows.append((img_id, pred))

rows.sort(key=lambda r: r[0])  # ascending ID order

OUTPUT_CSV = 'submission.csv'
with open(OUTPUT_CSV, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID', 'Label'])
    writer.writerows(rows)

print(f"Wrote {len(rows)} predictions to {OUTPUT_CSV}")

## 15. Verify the Submission

A quick sanity check before uploading:
- Exactly 1000 rows (one per test image)
- Labels are integers in `[0, 99]`
- No missing values

In [ ]:
import pandas as pd

sub = pd.read_csv(OUTPUT_CSV)
print(sub.head(10))
print(f"\nShape        : {sub.shape}")
print(f"Label range  : {sub['Label'].min()} – {sub['Label'].max()}")
print(f"Missing vals : {sub.isnull().sum().sum()}")
assert len(sub) == 1000, "Expected 1000 rows!"
assert sub['Label'].between(0, 99).all(), "Labels out of range!"
print("\nAll checks passed — ready to submit.")